# Module C2 — Déboguer avec le logger

**Phase 3 — Savoirs transversaux**  
Compatible Google Colab : logger simulé en mémoire (liste de chaînes).

## Section 1 — Rappels synthétiques

### Déboguer = rendre l'état visible
Un bug = écart entre état **attendu** et état **réel**. On débogue en affichant l'état réel à chaque étape.

### Les 3 outils
| Outil | Usage | Avantage |
|---|---|---|
| `print()` | Débogage rapide, temporaire | Simple, immédiat |
| Logger (fichier) | Traces persistantes, production | Lisible après crash |
| `assert` | Vérification d'état attendu | Stoppe si écart détecté |

### Lire une stack trace
```
Traceback (most recent call last):
  File "console.py", line 42, in phase_j1
    resultat = executer_mouvement(etat, id_drone, col, lig)
  File "logique.py", line 187, in executer_mouvement
    etat['drones'][id_drone]['col'] = col
KeyError: 'D4'
```
**Lecture** : l'erreur est dans `logique.py` ligne 187. `'D4'` n'existe pas dans `etat['drones']`.

### Rappel du module précédent (C1)
`logger.py` n'importe rien et n'a aucune dépendance. Il expose `demarrer_log()`, `ecrire_log()`, `fermer_log()`.

## Section 2 — Exercice guidé : trouver 3 bugs avec le logger

Le code ci-dessous contient **3 bugs intentionnels**. Tu vas les identifier en étudiant les traces du logger.

### Configuration initiale — Logger simulé en mémoire

In [ ]:
# Logger simulé (compatible Colab : pas de fichier)
_log = []

def ecrire_log(message):
    _log.append(message)

def afficher_log():
    for ligne in _log:
        print(ligne)

# État initial de test
etat = {
    'tour': 1,
    'score': 0,
    'partie_finie': False,
    'victoire': False,
    'grille': [['.' for _ in range(12)] for _ in range(12)],
    'hopital': (0, 0),
    'batiments': [],
    'drones': {
        'D1': {'id': 'D1', 'col': 2, 'lig': 3, 'batterie': 8, 'batterie_max': 10,
               'survivant': None, 'bloque': 0, 'hors_service': False},
    },
    'tempetes': {
        'T1': {'id': 'T1', 'col': 5, 'lig': 5, 'dx': 1, 'dy': 0},
    },
    'survivants': {
        'S1': {'id': 'S1', 'col': 3, 'lig': 3, 'etat': 'en_attente'},
    },
    'zones_x': set(),
    'historique': [],
    'nb_tours_max': 30,
}

### Étape 1 — Code buggé : identifier les 3 bugs

In [ ]:
def executer_mouvement_bugge(etat, id_drone, nouvelle_col, nouvelle_lig):
    """Version buggée d'executer_mouvement. Trouve les 3 bugs !"""
    drone = etat['drones'][id_drone]
    
    # Bug potentiel 1 : mise à jour de la position
    ecrire_log(f"[Tour {etat['tour']}] Avant : {id_drone} en ({drone['col']}, {drone['lig']})")
    drone['col'] = nouvelle_lig   # BUG 1 : col et lig inversés
    drone['lig'] = nouvelle_col   # BUG 1 suite
    ecrire_log(f"[Tour {etat['tour']}] Après : {id_drone} en ({drone['col']}, {drone['lig']})")
    
    # Bug potentiel 2 : consommation batterie
    cout = 1
    ecrire_log(f"[Tour {etat['tour']}] Coût calculé : {cout}")
    drone['batterie'] += cout     # BUG 2 : += au lieu de -=
    ecrire_log(f"[Tour {etat['tour']}] Batterie après : {drone['batterie']}")
    
    # Bug potentiel 3 : vérification de victoire
    nb_sauves = sum(
        1 for s in etat['survivants'].values()
        if s['etat'] == 'sauve'
    )
    ecrire_log(f"[Tour {etat['tour']}] Survivants sauvés : {nb_sauves}")
    # BUG 3 : condition inversée (len au lieu de nb_sauves)
    if len(etat['survivants']) == len(etat['survivants']):
        etat['victoire'] = True
        etat['partie_finie'] = True
    
    return f"{id_drone} déplacé"

### Étape 2 — Exécuter et lire les traces

In [ ]:
# Exécuter le mouvement buggé et inspecter les traces
executer_mouvement_bugge(etat, 'D1', 4, 3)  # on veut aller en (col=4, lig=3)

print('=== TRACES DU LOGGER ===')
afficher_log()

print()
print(f"Position finale D1 : col={etat['drones']['D1']['col']}, lig={etat['drones']['D1']['lig']}")
print(f"Batterie D1 : {etat['drones']['D1']['batterie']}")
print(f"Partie finie : {etat['partie_finie']}")

# TODO : en lisant les traces, identifie les 3 bugs et note-les ci-dessous :
# Bug 1 : ...
# Bug 2 : ...
# Bug 3 : ...

### Étape 3 — Corriger les 3 bugs

In [ ]:
# Réinitialiser l'état
etat['drones']['D1'].update({'col': 2, 'lig': 3, 'batterie': 8})
etat['partie_finie'] = False
etat['victoire'] = False
_log.clear()

def executer_mouvement_corrige(etat, id_drone, nouvelle_col, nouvelle_lig):
    """Version corrigée d'executer_mouvement."""    drone = etat['drones'][id_drone]
    
    ecrire_log(f"[Tour {etat['tour']}] Avant : {id_drone} en ({drone['col']}, {drone['lig']})")
    # TODO : corriger Bug 1 (col et lig inversés)
    drone['col'] = None  # TODO
    drone['lig'] = None  # TODO
    ecrire_log(f"[Tour {etat['tour']}] Après : {id_drone} en ({drone['col']}, {drone['lig']})")
    
    cout = 1
    # TODO : corriger Bug 2 (+= au lieu de -=)
    drone['batterie'] = None  # TODO
    ecrire_log(f"[Tour {etat['tour']}] Batterie après : {drone['batterie']}")
    
    nb_sauves = sum(1 for s in etat['survivants'].values() if s['etat'] == 'sauve')
    # TODO : corriger Bug 3 (condition toujours vraie)
    if None:  # TODO : bonne condition de victoire ?
        etat['victoire'] = True
        etat['partie_finie'] = True
    
    return f"{id_drone} déplacé"

executer_mouvement_corrige(etat, 'D1', 4, 3)

# Vérifications
assert etat['drones']['D1']['col'] == 4, f"col attendu 4, obtenu {etat['drones']['D1']['col']}"
assert etat['drones']['D1']['lig'] == 3, f"lig attendu 3, obtenu {etat['drones']['D1']['lig']}"
assert etat['drones']['D1']['batterie'] == 7, f"batterie attendue 7, obtenue {etat['drones']['D1']['batterie']}"
assert etat['partie_finie'] == False, "La partie ne doit pas être finie (aucun survivant sauve")
print('✅ Les 3 bugs corrigés !')

➡️ **Corrigé** : voir `C2_corrige.py`  
Module suivant : **C3 — Versionner avec Git**